In [1]:
# ============================================================
# 2D NEUTRON DIFFUSION SIMULATION
# WITH MP4 VIDEO GENERATION
# ============================================================
#
# Author: OpenAI ChatGPT
#
# Solves the time-dependent 2D neutron diffusion equation
# using finite differences and exports the animation to MP4.
#
# Output:
#     neutron_diffusion.mp4
#
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
import time

# ============================================================
# SIMULATION PARAMETERS
# ============================================================

Nx = 120
Ny = 120

Lx = 200.0      # cm
Ly = 200.0      # cm

dx = Lx / Nx
dy = Ly / Ny

dt = 0.01
steps = 700

# ============================================================
# NUCLEAR PARAMETERS
# ============================================================

D = 1.2

Sigma_a = 0.08
Sigma_f = 0.10

nu = 2.4

production = nu * Sigma_f - Sigma_a

# ============================================================
# COMPUTATIONAL GRID
# ============================================================

x = np.linspace(-Lx/2, Lx/2, Nx)
y = np.linspace(-Ly/2, Ly/2, Ny)

X, Y = np.meshgrid(x, y)

# ============================================================
# INITIAL NEUTRON FLUX
# ============================================================

phi = np.zeros((Ny, Nx))

source_radius = 20.0

source_region = X**2 + Y**2 < source_radius**2

phi[source_region] = 1.0

# ============================================================
# MATERIAL MAPS
# ============================================================

R = np.sqrt(X**2 + Y**2)

D_map = D * np.ones_like(phi)
Sigma_a_map = Sigma_a * np.ones_like(phi)
production_map = production * np.ones_like(phi)

# Reflector

reflector = R > 70

D_map[reflector] = 1.8
Sigma_a_map[reflector] = 0.02
production_map[reflector] = -0.01

# ============================================================
# LAPLACIAN OPERATOR
# ============================================================

def laplacian(Z):

    Ztop = np.roll(Z, -1, axis=0)
    Zbottom = np.roll(Z, 1, axis=0)

    Zleft = np.roll(Z, -1, axis=1)
    Zright = np.roll(Z, 1, axis=1)

    return (
        (Zleft + Zright - 2*Z)/dx**2
        +
        (Ztop + Zbottom - 2*Z)/dy**2
    )

# ============================================================
# BOUNDARY CONDITIONS
# ============================================================

def apply_boundary(Z):

    Z[0, :] = 0
    Z[-1, :] = 0

    Z[:, 0] = 0
    Z[:, -1] = 0

    return Z

# ============================================================
# PRECOMPUTE SOLUTION
# ============================================================

print("Running neutron diffusion simulation...")

start_time = time.time()

frames_data = []

for step in range(steps):

    lap_phi = laplacian(phi)

    dphi_dt = (
        D_map * lap_phi
        +
        production_map * phi
    )

    phi = phi + dt * dphi_dt

    phi = np.clip(phi, 0.0, None)

    phi = apply_boundary(phi)

    max_flux = phi.max()

    if max_flux > 0:
        phi = phi / max_flux

    if step % 2 == 0:
        frames_data.append(phi.copy())

elapsed = time.time() - start_time

print(f"Simulation completed in {elapsed:.2f} s")
print(f"Frames stored: {len(frames_data)}")

# ============================================================
# VISUALIZATION
# ============================================================

fig, ax = plt.subplots(figsize=(8,7))

im = ax.imshow(
    frames_data[0],
    cmap="inferno",
    origin="lower",
    extent=[-Lx/2, Lx/2, -Ly/2, Ly/2],
    vmin=0,
    vmax=1
)

cbar = plt.colorbar(im)
cbar.set_label("Normalized Neutron Flux")

ax.set_xlabel("x [cm]")
ax.set_ylabel("y [cm]")

# ============================================================
# ANIMATION FUNCTION
# ============================================================

def animate(i):

    im.set_array(frames_data[i])

    ax.set_title(
        f"2D Neutron Diffusion\n"
        f"Frame {i+1}/{len(frames_data)}"
    )

    return [im]

# ============================================================
# CREATE ANIMATION
# ============================================================

anim = FuncAnimation(
    fig,
    animate,
    frames=len(frames_data),
    interval=30,
    blit=True
)

# ============================================================
# EXPORT MP4
# ============================================================

print("Generating MP4 file...")

writer = FFMpegWriter(
    fps=30,
    metadata={
        "title": "2D Neutron Diffusion Simulation",
        "artist": "Python"
    },
    bitrate=2500
)

video_filename = "neutron_diffusion.mp4"

anim.save(
    video_filename,
    writer=writer,
    dpi=150
)

print()
print("========================================")
print("VIDEO SUCCESSFULLY GENERATED")
print("========================================")
print(f"File : {video_filename}")
print("========================================")

# ============================================================
# FINAL STATISTICS
# ============================================================

print()
print("Final Flux Statistics")
print("----------------------")
print(f"Maximum Flux : {phi.max():.6f}")
print(f"Average Flux : {phi.mean():.6f}")
print(f"Grid         : {Nx} x {Ny}")
print(f"Iterations   : {steps}")

plt.close()

Running neutron diffusion simulation...
Simulation completed in 0.12 s
Frames stored: 350
Generating MP4 file...

VIDEO SUCCESSFULLY GENERATED
File : neutron_diffusion.mp4

Final Flux Statistics
----------------------
Maximum Flux : 1.000000
Average Flux : 0.031112
Grid         : 120 x 120
Iterations   : 700
